# Nutrients Correlation
Our collaborators have processed shells and tissues from oysters in the same tanks/treatments (not the same oysters) for % nitrogen and carbon. I'm going to attempt to correlate these values with gene expression to connect gene activity with nutrient bioassimilation (is that the right word?)


hmmm but actually in thinking about this more, I'm not sure this will be possible - since we don't have nutrient data for the sequenced oysters, the most I can do is a treatment-level association. So the best approach might be to understand which nutrients are affected by treatments and how, and find similar patterns in gene expression (that respond in similar ways) and see if there are any affected pathways that might make sense together, rather than directly trying to correlate the two datasets

## 0. load libraries

In [2]:
library(tidyverse)

## 1. read CSVs

### phase 1

#### vst

In [5]:
p1.vst <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/analysis/diff_expression/phase1_v_phase1/deseq_res/prefilter_deseq/p1_vst.csv') %>%
rename('Gene' = X)
head(p1.vst)

,Gene,B1_Nu_O03,B2_Nu_O12,B4_Nu_O32,B5_Nu_O36,B6_Nu_O47,C1_Nu_W01,C1_Nu_W05,C2_Nu_W15,C3_Nu_W21,⋯,H3_Nu_B18,H4_Nu_B28,H5_Nu_B35,H6_Nu_B45,W1_Nu_G02,W2_Nu_G15,W3_Nu_G21,W4_Nu_G27,W6_Nu_G41,W6_Nu_G45
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,LOC144621260,11.267274,11.965325,11.273842,11.430104,11.532611,11.376177,11.656856,11.637867,10.994990,⋯,11.731597,11.209268,11.695066,12.028253,11.414145,11.525916,11.209461,11.734460,11.505541,11.635744
2,LOC144621269,9.410200,14.240105,13.261172,7.329003,7.964236,13.081891,11.300252,11.691961,11.731097,⋯,13.185637,11.286698,8.636007,15.537217,13.126994,11.623331,11.907070,13.452947,6.829172,12.364252
3,LOC111120925,6.893306,7.223246,7.309465,6.844029,7.046606,7.258233,9.081128,9.102962,8.573904,⋯,9.137864,7.172068,8.717478,8.615532,7.000773,9.012082,6.966731,6.993037,6.624733,7.161980
4,LOC144621283,10.007130,10.255247,10.357057,10.068160,10.456028,10.154534,9.858656,10.256025,10.193238,⋯,10.193226,9.925137,9.561564,9.951248,9.462986,10.647775,10.653863,9.789103,10.216260,10.245299
5,LOC144621276,13.551204,13.774916,14.152285,13.237453,13.098945,13.059040,13.818545,13.606651,13.321595,⋯,13.937574,13.695373,12.894043,14.004262,12.976861,13.900594,14.187734,13.345133,13.936585,13.560586
6,LOC111115920,8.948539,8.728712,8.632308,9.157246,9.472853,8.734235,8.266068,8.462519,8.638520,⋯,7.983656,8.471196,8.925540,8.351976,8.169439,8.347215,8.705016,8.411923,8.451989,8.149142


In [7]:
p1.vst.long <- p1.vst %>%
pivot_longer(cols = -Gene,
              values_to = 'vst',
              names_to = 'sample')

head(p1.vst.long)

Gene,sample,vst
<chr>,<chr>,<dbl>
LOC144621260,B1_Nu_O03,11.26727
LOC144621260,B2_Nu_O12,11.96532
LOC144621260,B4_Nu_O32,11.27384
LOC144621260,B5_Nu_O36,11.43010
LOC144621260,B6_Nu_O47,11.53261
LOC144621260,C1_Nu_W01,11.37618


#### nutrients

In [3]:
p1.nutes <- read.csv('/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/metaData/nutrients/Phase1_nutrient_merged.csv')
head(p1.nutes)

,Sample_Name,Order_pre,Date_MMDDYYYY_pre,Year,Phase_1_treat,Phase_1_rep,Phase_2_treat,Phase_2_rep,Tag_color,Tag_num,⋯,Phase_1_rep_R,Phase_2_rep_R,Ratio_tissue_shell_pre_mg,Phase1_Phase2_rep,Exclude_all,Exclude_pre_analysis,Phase1N_tissue,Phase1C_tissue,Phase1N_shell,Phase1C_shell
,<chr>,<int>,<chr>,<int>,<chr>,<int>,<chr>,<int>,<chr>,<int>,⋯,<chr>,<chr>,<dbl>,<chr>,<chr>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2024_Both01_Both01_O1,3,7/23/24,2024,Both,1,Both,1,O,1,⋯,Both01,Both01,0.9025393,Both01_Both01,NA,NA,21.16298,118.95000,0.99388,55.38134
2,2024_Both01_Both01_O2,7,7/23/24,2024,Both,1,Both,1,O,2,⋯,Both01,Both01,0.6425681,Both01_Both01,NA,NA,15.65760,88.00615,1.03283,57.55181
3,2024_Both01_Both02_O13,50,7/23/24,2024,Both,1,Both,2,O,13,⋯,Both01,Both02,0.8083796,Both01_Both02,NA,NA,17.63241,99.10589,0.92453,51.51684
4,2024_Both01_Both02_O14,55,7/23/24,2024,Both,1,Both,2,O,14,⋯,Both01,Both02,0.5888140,Both01_Both02,NA,NA,15.42337,86.68961,1.11026,61.86629
5,2024_Both01_Both03_O25,98,7/24/24,2024,Both,1,Both,3,O,25,⋯,Both01,Both03,0.7725658,Both01_Both03,NA,NA,6.44419,36.22065,0.35356,19.70089
6,2024_Both01_Both03_O26,103,7/24/24,2024,Both,1,Both,3,O,26,⋯,Both01,Both03,0.5599379,Both01_Both03,NA,NA,11.03365,62.01647,0.83523,46.54063
